# VAE -- Decoder and Training

In this notebook we build the decoder and put the full VAE together.

The decoder takes the latent vector `z` and generates a stroke sequence autoregressively -- one step at a time, feeding its own output back as the next input. We then train the full encoder-decoder end to end.

This is **Checkpoint 2: Latent Compression**. By the end, your model should be able to compress a sketch into `z` and reconstruct a recognisable doodle from it.

## 0. Setup

Run this cell to import everything from the previous notebooks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json, os
import matplotlib.pyplot as plt
import urllib.request
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

# ---- data pipeline ----
def drawing_to_stroke3(drawing):
    strokes = []
    for stroke in drawing:
        xs, ys = stroke[0], stroke[1]
        for i in range(len(xs)):
            dx = xs[i] - xs[i-1] if i > 0 else 0
            dy = ys[i] - ys[i-1] if i > 0 else 0
            pen_lifted = 1 if i == len(xs) - 1 else 0
            strokes.append([dx, dy, pen_lifted])
    return np.array(strokes, dtype=np.float32)

def normalise_stroke3(stroke3):
    s = stroke3.copy()
    coords = s[:, :2]
    s[:, :2] = (coords - coords.mean(axis=0)) / (coords.std(axis=0) + 1e-8)
    return s

class QuickDrawDataset(Dataset):
    def __init__(self, path, max_len=200, max_samples=5000):
        self.samples = []
        with open(path) as f:
            for i, line in enumerate(f):
                if i >= max_samples: break
                d  = json.loads(line)
                s3 = drawing_to_stroke3(d['drawing'])
                s3 = normalise_stroke3(s3)
                if len(s3) > max_len: s3 = s3[:max_len]
                self.samples.append(torch.tensor(s3, dtype=torch.float32))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def collate_fn(batch):
    lengths = [seq.shape[0] for seq in batch]
    padded  = pad_sequence(batch, batch_first=True, padding_value=0.0)
    return padded, lengths

# ---- encoder ----
class Encoder(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=256, latent_dim=128):
        super().__init__()
        self.lstm      = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc_mu     = nn.Linear(hidden_dim * 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim * 2, latent_dim)
    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h = torch.cat([h[0], h[1]], dim=-1)
        return self.fc_mu(h), self.fc_logvar(h)

def reparameterise(mu, logvar):
    std = torch.exp(0.5 * logvar)
    return mu + torch.randn_like(std) * std

def kl_loss(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

# ---- data ----
os.makedirs('data', exist_ok=True)
path = 'data/cat.ndjson'
if not os.path.exists(path):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/quickdraw_dataset/full/simplified/cat.ndjson', path)

dataset = QuickDrawDataset(path, max_len=200, max_samples=3000)
loader  = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
print(f'Dataset: {len(dataset)} samples')
print('Setup complete.')


## 1. The Decoder

The decoder is an autoregressive LSTM. It takes `z` and generates the stroke sequence one step at a time:

1. Initialise the LSTM hidden state from `z` using a linear layer
2. Start with a zero input token (the start-of-sequence token)
3. At each step, predict the next `(dx, dy, pen_lifted)` and feed it back as the next input
4. Repeat until the max sequence length is reached

During training we use **teacher forcing**: instead of feeding the model's own prediction back, we feed the ground truth previous step. This makes training more stable.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=128, hidden_dim=512, output_dim=3):
        '''
        latent_dim : size of z
        hidden_dim : size of the decoder LSTM hidden state
        output_dim : 3 for stroke-3 (dx, dy, pen_lifted)
        '''
        super().__init__()
        # Project z into the initial hidden and cell states of the LSTM
        self.fc_hidden = nn.Linear(latent_dim, hidden_dim)
        self.fc_cell   = nn.Linear(latent_dim, hidden_dim)

        self.lstm = nn.LSTM(
            input_size=output_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, z, target_seq):
        '''
        Teacher-forced forward pass (used during training).

        z          : latent vector, shape (batch, latent_dim)
        target_seq : ground truth sequence, shape (batch, seq_len, 3)
                     we shift it right by one step to use as decoder input

        Returns:
            output : predicted sequence, shape (batch, seq_len, 3)
        '''
        batch_size = z.shape[0]

        # Initialise hidden state from z
        h0 = torch.tanh(self.fc_hidden(z)).unsqueeze(0)   # (1, batch, hidden)
        c0 = torch.tanh(self.fc_cell(z)).unsqueeze(0)

        # Shift target right: prepend a zero start token, drop the last step
        start_token = torch.zeros(batch_size, 1, 3, device=z.device)
        decoder_input = torch.cat([start_token, target_seq[:, :-1, :]], dim=1)

        output, _ = self.lstm(decoder_input, (h0, c0))
        output = self.fc_out(output)   # (batch, seq_len, 3)
        return output

    def generate(self, z, max_len=200):
        '''
        Autoregressive generation (used during inference).
        Feed each predicted step back as the next input.
        '''
        batch_size = z.shape[0]
        h = torch.tanh(self.fc_hidden(z)).unsqueeze(0)
        c = torch.tanh(self.fc_cell(z)).unsqueeze(0)

        input_step = torch.zeros(batch_size, 1, 3, device=z.device)
        outputs = []

        for _ in range(max_len):
            out, (h, c) = self.lstm(input_step, (h, c))
            step = self.fc_out(out)           # (batch, 1, 3)
            outputs.append(step)
            input_step = step                 # feed prediction back

        return torch.cat(outputs, dim=1)      # (batch, max_len, 3)

decoder = Decoder(latent_dim=128, hidden_dim=512)
print(decoder)


In [ ]:
# TODO: Why do we use teacher forcing during training but not during generation?
#       What could go wrong if we used the model's own predictions during training?
# YOUR CODE HERE

# Teacher forcing provides the ground truth as the next input, keeping training stable and fast. 
# During generation, we don't have ground truth, so the model must feed its own predictions back to itself.
# If we used the model's own predictions during early training, it would predict garbage, feed that garbage back in, and the errors would compound so badly that the model would struggle to learn anything useful.


## 2. The full VAE

Now we put encoder and decoder together into a single `SketchVAE` module.

In [ ]:
class SketchVAE(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=256, latent_dim=128, decoder_hidden=512):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, decoder_hidden, input_dim)

    def forward(self, x, lengths):
        mu, logvar = self.encoder(x, lengths)
        z          = reparameterise(mu, logvar)
        recon      = self.decoder(z, x)
        return recon, mu, logvar

model = SketchVAE()
total = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total:,}')

# Quick forward pass test
padded, lengths = next(iter(loader))
recon, mu, logvar = model(padded, lengths)
print('Input shape      :', padded.shape)
print('Reconstruction   :', recon.shape)
print('mu shape         :', mu.shape)


## 3. The Loss Function

The VAE loss has two terms:

- **Reconstruction loss:** MSE between the predicted and actual `(dx, dy)`. For `pen_lifted` we use binary cross-entropy since it is a 0/1 value.
- **KL loss:** pushes the latent distribution toward N(0, 1).

We weight the KL term with a scalar `kl_weight` (also called beta). Starting with a small value and increasing it during training (KL annealing) helps the model learn a good reconstruction before worrying about the latent structure.

In [ ]:
def vae_loss(recon, target, mu, logvar, kl_weight=0.5):
    '''
    recon    : predicted sequence (batch, seq_len, 3)
    target   : ground truth sequence (batch, seq_len, 3)
    mu       : latent mean
    logvar   : latent log-variance
    kl_weight: weight on the KL term
    '''
    # MSE on position deltas
    recon_pos  = F.mse_loss(recon[:, :, :2], target[:, :, :2])

    # BCE on pen state
    pen_pred   = torch.sigmoid(recon[:, :, 2])
    recon_pen  = F.binary_cross_entropy(pen_pred, target[:, :, 2], reduction='mean')

    kl         = kl_loss(mu, logvar)

    total      = recon_pos + recon_pen + kl_weight * kl
    return total, recon_pos, recon_pen, kl

# Test loss
loss, rp, rpen, kl = vae_loss(recon, padded, mu, logvar)
print(f'Total loss  : {loss.item():.4f}')
print(f'Recon pos   : {rp.item():.4f}')
print(f'Recon pen   : {rpen.item():.4f}')
print(f'KL loss     : {kl.item():.4f}')


## 4. Training loop

In [ ]:
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
model     = SketchVAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 20
history    = {'total': [], 'recon_pos': [], 'recon_pen': [], 'kl': []}

for epoch in range(num_epochs):
    model.train()
    epoch_losses = {'total': 0, 'recon_pos': 0, 'recon_pen': 0, 'kl': 0}

    # KL annealing: start at 0, linearly increase to 0.5 over training
    kl_weight = min(0.5, epoch / num_epochs)

    for padded, lengths in loader:
        padded = padded.to(device)

        recon, mu, logvar = model(padded, lengths)
        loss, rp, rpen, kl = vae_loss(recon, padded, mu, logvar, kl_weight)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()

        epoch_losses['total']     += loss.item()
        epoch_losses['recon_pos'] += rp.item()
        epoch_losses['recon_pen'] += rpen.item()
        epoch_losses['kl']        += kl.item()

    n = len(loader)
    for k in history:
        history[k].append(epoch_losses[k] / n)

    print(f'Epoch {epoch+1:02d}/{num_epochs} | '
          f'loss {history["total"][-1]:.4f} | '
          f'recon_pos {history["recon_pos"][-1]:.4f} | '
          f'kl {history["kl"][-1]:.4f} | '
          f'kl_weight {kl_weight:.2f}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(history['total']);     axes[0].set_title('Total loss')
axes[1].plot(history['recon_pos']); axes[1].set_title('Reconstruction loss (position)')
axes[2].plot(history['kl']);        axes[2].set_title('KL loss')
for ax in axes:
    ax.set_xlabel('Epoch')
plt.tight_layout()
plt.show()


## 5. Visualising reconstructions

Let's see how well the model reconstructs sketches it has seen during training.

In [ ]:
def stroke3_to_absolute(stroke3):
    abs_coords = np.cumsum(stroke3[:, :2], axis=0)
    pen_lifted  = stroke3[:, 2]
    return abs_coords, pen_lifted

def plot_sketch(stroke3_np, title='', ax=None):
    coords, pen_lifted = stroke3_to_absolute(stroke3_np)
    if ax is None:
        fig, ax = plt.subplots(figsize=(3, 3))
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off'); ax.set_title(title, fontsize=9)
    start = 0
    for i in range(len(pen_lifted)):
        if pen_lifted[i] > 0.5:
            seg = coords[start : i + 1]
            ax.plot(seg[:, 0], seg[:, 1], 'k-', linewidth=1.5)
            start = i + 1

model.eval()
with torch.no_grad():
    sample_batch, sample_lengths = next(iter(loader))
    sample_batch = sample_batch.to(device)
    recon, mu, logvar = model(sample_batch, sample_lengths)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    original = sample_batch[i].cpu().numpy()
    reconstructed = recon[i].cpu().numpy()
    plot_sketch(original,      title=f'Original {i}',       ax=axes[0, i])
    plot_sketch(reconstructed, title=f'Reconstructed {i}',  ax=axes[1, i])

plt.suptitle('Top: original   Bottom: reconstructed', fontsize=11)
plt.tight_layout()
plt.show()


## 6. Generating new sketches from random z

Now the fun part -- sampling from the latent space to generate completely new sketches.

In [ ]:
model.eval()
with torch.no_grad():
    z_random = torch.randn(8, 128).to(device)   # sample 8 random latent vectors
    generated = model.decoder.generate(z_random, max_len=150)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    plot_sketch(generated[i].cpu().numpy(), title=f'Generated {i}', ax=ax)

plt.suptitle('Sketches generated from random z', fontsize=11)
plt.tight_layout()
plt.show()

# TODO: Do the generated sketches look like cats? If not, why might that be?
#       What would you change to improve the quality?
# YOUR CODE HERE

# They likely look a bit like messy squiggles or very abstract blobs rather than perfect cats.
# This happens because our model is small, trained for very few epochs, and only saw 3000 training samples.
# To improve quality, we need to train on the full dataset (hundreds of thousands of cats), increase the hidden dimensions, train for much longer, and potentially use a more advanced architecture like a Transformer or an LSTM with attention.


## 7. Latent space interpolation

One of the nicest properties of a VAE is that the latent space is smooth and continuous. We can interpolate between two sketches by linearly blending their `z` vectors.

In [ ]:
model.eval()
with torch.no_grad():
    # Encode two sketches
    s1 = dataset[0].unsqueeze(0).to(device)
    s2 = dataset[1].unsqueeze(0).to(device)
    mu1, _ = model.encoder(s1, [s1.shape[1]])
    mu2, _ = model.encoder(s2, [s2.shape[1]])

    # Interpolate between mu1 and mu2 in 6 steps
    steps = 6
    alphas = torch.linspace(0, 1, steps)
    z_interp = torch.stack([alpha * mu2 + (1 - alpha) * mu1 for alpha in alphas]).squeeze(1)

    generated = model.decoder.generate(z_interp, max_len=150)

fig, axes = plt.subplots(1, steps, figsize=(3 * steps, 3))
for i, ax in enumerate(axes):
    plot_sketch(generated[i].cpu().numpy(),
                title=f'alpha={alphas[i]:.1f}', ax=ax)

plt.suptitle('Latent interpolation between two cat sketches', fontsize=11)
plt.tight_layout()
plt.show()

# TODO: Does the interpolation look smooth? What does smoothness here mean for the model?
# YOUR CODE HERE

# Yes, the transition should look relatively smooth, morphing gradually from the first cat into the second.
# Smoothness here means that our latent space is well-regularized (thanks to the KL loss) and continuous. 
# It proves that the model hasn't just memorized individual drawings, but has learned a meaningful geometric space where points close together decode to visually similar sketches.


## 8. Save the model

In [ ]:
torch.save(model.state_dict(), 'sketch_vae.pth')
print('Model saved to sketch_vae.pth')
print('We will load this in the next set of notebooks when we build the hierarchical architecture.')


## Checkpoint 2 complete -- Latent Compression!

You have built a full VAE that:
- Encodes a sketch sequence into a latent vector z
- Reconstructs the sketch from z
- Generates new sketches from random z
- Interpolates smoothly between two sketches in latent space